# 06 - Audio Roundtrip (Speech In, Speech Out)

This notebook transcribes a customer voice message, creates a triage response, and synthesizes that response back to audio.

Pre-requisite: run `05_audio_text.ipynb` first.

---

## Step 1 – Load environment variables

In [1]:
import json
import os
from pathlib import Path

import requests
import azure.cognitiveservices.speech as speechsdk
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

endpoint = os.environ["AZURE_AI_ENDPOINT"].rstrip("/")
deployment = os.environ["AZURE_AI_DEPLOYMENT"]
location = os.environ["AZURE_LOCATION"]
voice = os.environ.get("AZURE_SPEECH_VOICE", "en-US-JennyNeural")
auth_mode = os.environ.get("AZURE_AUTH_MODE", "aad").lower()
if auth_mode != "aad":
    raise RuntimeError(f"Unsupported auth mode for this demo: {auth_mode}. Expected 'aad'.")

credential = AzureCliCredential()
credential.get_token("https://cognitiveservices.azure.com/.default")
credential_source = "AzureCliCredential"
audio_path = Path(os.environ["LOCAL_AUDIO_PATH"]).expanduser()

if not audio_path.exists():
    raise FileNotFoundError(f"Audio file not found: {audio_path}")

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

print(f"Audio path        : {audio_path.resolve()}")
print(f"Voice             : {voice}")
print(f"Credential source : {credential_source}")

Audio path        : C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\gpt4o01-text-image-audio\data\goodicecream.wav
Voice             : en-US-JennyNeural
Credential source : AzureCliCredential


## Step 2 – Create clients / connections

Initialise any SDK clients using the loaded environment variables.

In [2]:
speech_token = credential.get_token("https://cognitiveservices.azure.com/.default").token
speech_endpoint = os.environ["AZURE_AI_ENDPOINT"]

speech_config = speechsdk.SpeechConfig(endpoint=speech_endpoint)
speech_config.authorization_token = speech_token
speech_config.speech_recognition_language = "en-US"
audio_input = speechsdk.audio.AudioConfig(filename=str(audio_path))
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_input)

recognized = recognizer.recognize_once_async().get()
if recognized.reason != speechsdk.ResultReason.RecognizedSpeech:
    raise RuntimeError(f"Speech recognition failed: {recognized.reason}")

transcript = recognized.text
print("Transcript:")
print(transcript)

Transcript:
Mooth, texture, balance, sweetness and justice. The right level of chill. It quietly does everything you'd want without trying too hard. The kind of thing you keep going back to, not out of habit, but because it genuinely makes the moment better.


## Step 3 – Baseline test

A minimal call to confirm the deployment is reachable.

In [3]:
url = f"{endpoint}/openai/deployments/{deployment}/chat/completions?api-version=2024-10-21"
messages = [
    {"role": "system", "content": "You are a concise support triage assistant."},
    {"role": "user", "content": f"Draft a short spoken response to this customer. Include empathy, sentiment classification, urgency level, and next step in under 60 words: {transcript}"},
]

model_token = credential.get_token("https://cognitiveservices.azure.com/.default").token
response = requests.post(
    url,
    headers={"Authorization": f"Bearer {model_token}", "Content-Type": "application/json"},
    json={"messages": messages, "max_completion_tokens": 220},
    timeout=90,
)
if response.status_code >= 400:
    raise RuntimeError(f"GPT call failed: {response.status_code} {response.text}")

body = response.json()
assistant_text = body["choices"][0]["message"]["content"]
print("Assistant text:")
print(assistant_text)

Assistant text:
Thank you for sharing such kind words! It’s clear you’re passionate and satisfied, which I’d classify as positive sentiment. This doesn’t appear urgent, but I’ll ensure your feedback reaches our team for acknowledgment and improvement. We're grateful it enhances your moments!


## Step 4 – Demo scenario

Implement the key scenario that validates the customer hypothesis.

In [4]:
speech_token = credential.get_token("https://cognitiveservices.azure.com/.default").token
speech_endpoint = os.environ["AZURE_AI_ENDPOINT"]

speech_config = speechsdk.SpeechConfig(endpoint=speech_endpoint)
speech_config.authorization_token = speech_token
speech_config.speech_synthesis_voice_name = voice

roundtrip_wav = output_dir / "06_audio_roundtrip_response.wav"
audio_output = speechsdk.audio.AudioOutputConfig(filename=str(roundtrip_wav))
synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_output)

synth_result = synthesizer.speak_text_async(assistant_text).get()
if synth_result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
    raise RuntimeError(f"Speech synthesis failed: {synth_result.reason}")

metadata = {
    "audio_input": str(audio_path),
    "transcript": transcript,
    "assistant_text": assistant_text,
    "wav_output": str(roundtrip_wav),
}
metadata_path = output_dir / "06_audio_roundtrip_result.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved audio: {roundtrip_wav.resolve()}")
print(f"Saved metadata: {metadata_path.resolve()}")

Saved audio: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\gpt4o01-text-image-audio\outputs\06_audio_roundtrip_response.wav
Saved metadata: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\gpt4o01-text-image-audio\outputs\06_audio_roundtrip_result.json


---

## Roundtrip checkpoint complete

Local staged flow is complete. If needed, run **X_destroy.ipynb** to clean up resources.